# **Low-Light Image Enhancement with Cross-Dataset Generalization**

#### **Preliminary Notebook Operations**

- Settings >> Turn on Internet (*if it is not already enabled*)
- Settings >> Accelerator >> **GPU P100**

Use the boolean flags in the following cell to enable or disable each stage of the pipeline. Set them according to whether you want to use the default preprocessed datasets or regenerate the data artifacts, and according to the type of experiment you want to run. For information beyond the basic usage of this notebook, refer to the repository README.

In [ ]:
RUN_MANIFEST_GENERATION = False
RUN_OFFLINE_PREPROCESSING = False
RUN_TRAINING_EVALUATION = True
RUN_LOG_PARSING = False

#### **Initialization**

In [ ]:
import os
import subprocess

from pathlib import Path

RELEASE_BASE_URL = (
  "https://github.com/cosimo-curri/"
  "cosimo-curri-594317-2026_III/"
  "releases/download/v1.0.0"
)

def run_streamed(command):
  env = os.environ.copy()
  env["PYTHONUNBUFFERED"] = "1"

  process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
  )

  try:
    assert process.stdout is not None

    for line in process.stdout:
      print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
      raise subprocess.CalledProcessError(return_code, command)

  except Exception:
    process.kill()
    process.wait()

    raise

!git clone https://github.com/cosimo-curri/cosimo-curri-594317-2026_III.git
%cd cosimo-curri-594317-2026_III

VENV_DIR = Path("/kaggle/working/lle_venv")
VENV_PYTHON = VENV_DIR / "bin" / "python"

%pip install virtualenv

if not VENV_PYTHON.exists():
  subprocess.run(
    ["python", "-m", "virtualenv", str(VENV_DIR)],  # full requirements compatibility
    check=True
  )

subprocess.run(
  [str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"],
  check=True
)

subprocess.run(
  [str(VENV_PYTHON), "-m", "pip", "install", "-r", "requirements.txt"],
  check=True
)

# Pre-download pyiqa weights used by NIQE and BRISQUE
# This keeps automatic downloads out of the experiment logs
subprocess.run(
  [
    str(VENV_PYTHON),
    "-c",
    (
      "import torch\n"
      "import pyiqa\n"
      "device = torch.device(\"cpu\")\n"
      "pyiqa.create_metric(\"niqe\", device=device)\n"
      "pyiqa.create_metric(\"brisque\", device=device)\n"
    )
  ],
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
  check=True
)

if RUN_MANIFEST_GENERATION or RUN_OFFLINE_PREPROCESSING:
  RAW_DIR = Path("data/raw")
  RAW_DIR.mkdir(parents=True, exist_ok=True)

  raw_archives = [
    f"{RELEASE_BASE_URL}/data-raw-1.zip",
    f"{RELEASE_BASE_URL}/data-raw-2.zip"
  ]

  for archive_url in raw_archives:
    archive_name = archive_url.split("/")[-1]

    subprocess.run(["wget", "-q", "-nc", archive_url], check=True)
    subprocess.run(["unzip", "-q", "-o", archive_name, "-d", str(RAW_DIR)], check=True)

    Path(archive_name).unlink()

#### **Manifest Generation**

If necessary, modify the configuration from the default version.

In [ ]:
if RUN_MANIFEST_GENERATION:
  path = "configs/data/datasets.yml"

  new_content = """\
manifests:
  raw_dir: data/raw
  output_dir: data/manifests

datasets:
  ex_dark:
    root: ex_dark
    output: ex_dark.csv

    splits:
      test:
        input_dir: .

    pairing:
      strategy: category_dirs

  lol_v1:
    root: lol_v1
    output: lol_v1.csv

    splits:
      test:
        sources:
          - input_dir: our485/low
            target_dir: our485/high
          - input_dir: eval15/low
            target_dir: eval15/high

    pairing:
      strategy: same_filename

  lol_v2_real:
    root: lol_v2/real_captured
    output: lol_v2_real.csv

    splits:
      train:
        input_dir: train/low
        target_dir: train/normal
      test:
        input_dir: test/low
        target_dir: test/normal

    validation:
      from_split: train
      fraction: 0.1
      seed: 42

    pairing:
      strategy: prefix_replace
      input_prefix: low
      target_prefix: normal

  lol_v2_synthetic:
    root: lol_v2/synthetic
    output: lol_v2_synthetic.csv

    splits:
      train:
        input_dir: train/low
        target_dir: train/normal
      test:
        input_dir: test/low
        target_dir: test/normal

    validation:
      from_split: train
      fraction: 0.1
      seed: 42

    pairing:
      strategy: same_filename

  mill_s:
    root: mill_s
    output: mill_s.csv

    splits:
      test:
        sources:
          - input_dir: train/input
            target_dir: train/gt
          - input_dir: validation/input
            target_dir: validation/gt
          - input_dir: test/input
            target_dir: test/gt

    pairing:
      strategy: suffix_level_to_gt
      separator: "-"
"""

  with open(path, "w") as f:
    f.write(new_content)

  subprocess.run(
    [
      str(VENV_PYTHON),
      "scripts/build_manifests.py",
      "--config",
      path
    ],
    check=True
  )

#### **Offline Preprocessing**

If necessary, modify the configuration from the default version.

In [ ]:
if RUN_OFFLINE_PREPROCESSING:
  path = "configs/data/preprocess.yml"

  new_content = """\
preprocessing:
  input_manifest_dir: data/manifests
  output_dir: data/preprocessed

  image:
    size: [256, 256]
    output_format: png

  statistics:
    dark_threshold: 0.10
"""

  with open(path, "w") as f:
    f.write(new_content)

  run_streamed(
		[
			str(VENV_PYTHON),
			"-u",
			"scripts/preprocess_offline.py",
			"--config",
			path,
			"--no-progress-bar"
		]
	)
else:
  PREPROCESSED_DIR = Path("data/preprocessed")
  PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

  preprocessed_archive_url = f"{RELEASE_BASE_URL}/data-preprocessed.zip"
  preprocessed_archive_name = preprocessed_archive_url.split("/")[-1]

  subprocess.run(["wget", "-q", "-nc", preprocessed_archive_url], check=True)

  subprocess.run(
    [
      "unzip",
      "-q",
      "-o",
      preprocessed_archive_name,
      "-d",
      str(PREPROCESSED_DIR)
    ],
    check=True
  )

  Path(preprocessed_archive_name).unlink()

#### **Training & Evaluation**

If necessary, modify the configuration path and content from the default version, depending on the model and experiment you want to run.

In [ ]:
if RUN_TRAINING_EVALUATION:
  path = "configs/experiments/unet.yml"

  new_content = """\
experiment:
  name: unet_grid_seed_0_screening
  seed: 0
  # restore_checkpoint:

data:
  train_manifest: data/preprocessed/lol_v2_real.csv  # for both training and validation
  test_manifests:  # only for evaluation.run_test: true
    - manifest: data/preprocessed/ex_dark.csv
      group_by: category
    - manifest: data/preprocessed/lol_v1.csv
    - manifest: data/preprocessed/lol_v2_real.csv
    - manifest: data/preprocessed/lol_v2_synthetic.csv
    - manifest: data/preprocessed/mill_s.csv
      group_by: illumination_level

augmentation:  # fixed online train-time augmentation
  enabled: true  # global switch (if false all augmentations are disabled)

  horizontal_flip:
    enabled: true
    p: 0.5

  random_resized_crop:
    enabled: true
    output_size: 256
    scale_min: 0.85
    scale_max: 1.0
    ratio_min: 0.95
    ratio_max: 1.05

  color_jitter:  # input-only
    enabled: true
    p: 0.3
    brightness_min: 0.9
    brightness_max: 1.1
    contrast_min: 0.9
    contrast_max: 1.1
    saturation_min: 0.95
    saturation_max: 1.05

  gamma_jitter:  # input-only
    enabled: true
    p: 0.3
    gamma_min: 0.85
    gamma_max: 1.15

  sensor_noise:  # input-only
    enabled: true
    p: 0.2
    sigma_min: 0.0
    sigma_max: 0.015

model:
  name: unet
  channels:  # the only case where a list is used for the grid
    - [16, 32, 64, 128]
    - [32, 64, 128, 256]
    - [32, 64, 128, 256, 512]
  normalization: [none, batch, instance, group]  # none | batch | instance | group
  activation: leaky_relu  # relu | leaky_relu
  group_count: 8  # used only by normalization: group
  upsampling: [bilinear, transposed]  # bilinear | transposed

optimizer:  # AdamW (beta1=0.9, beta2=0.999; they are not configurable)
  lr: [0.0003, 0.0005, 0.001]
  weight_decay: [0.0, 0.0001]

loss:
  name: l1_ssim  # SSIM window_size=11; it is not configurable
  l1_weight: 0.8  # weights must be non-negative and sum to 1.0
  ssim_weight: 0.2

training:
  epochs: 999
  validation_rate: 1
  batch_size: 16
  num_workers: 4
  mixed_precision: true  # true -> float32 & float16 | false -> float32
  deterministic: false  # true -> reproducibility | false -> efficiency

early_stopping:
  monitor: ssim  # window_size=11; it is not configurable
  mode_monitor: max
  min_delta: 0.0001
  patience: 15
  tie_breaker: psnr
  mode_tie_breaker: max

evaluation:
  run_test: false
"""

  with open(path, "w") as f:
    f.write(new_content)

  run_streamed(
		[
			str(VENV_PYTHON),
			"-u",
			"scripts/start_experiments.py",
			"--config",
			path,
			"--no-progress-bar"
		]
	)

  subprocess.run(["zip", "-q", "-r", "logs.zip", "logs"], check=True)
  subprocess.run(["zip", "-q", "-r", "checkpoints.zip", "checkpoints"], check=True)

#### **Log Parsing**

Run this section after a training/evaluation execution has produced the JSONL logs.

Alternatively, you can place an existing directory of JSONL logs from another training or evaluation run inside the project folder and parse those logs directly.

Before running the cell below, update:

- `parsing_mode`: parsing mode to execute (*see the comment in the next code cell*);
- `logs_dir`: path to the directory containing the JSONL log files;
- `csv_name`: name of the CSV file to generate inside `parsed_logs/`.

The parser will read all JSONL files found recursively in `logs_dir` and will produce a single CSV file according to the selected parsing mode.

In [ ]:
if RUN_LOG_PARSING:
  parsing_mode = "screening"  # screening | multi_seed | final_test | failure_candidates
  logs_dir = "experiment_logs/unet_grid_seed_0_screening"
  csv_name = "unet_grid_seed_0_screening.csv"

  run_streamed(
    [
      str(VENV_PYTHON),
      "-u",
      "scripts/parse_logs.py",
      "--parsing-mode",
      parsing_mode,
      "--logs-dir",
      logs_dir,
      "--csv-name",
      csv_name
    ]
  )

#### **Execution Instructions**

Save Version >> VERSION TYPE: Save & Run All (Commit) >> Save (*this allows background execution*)

#### **Results Visualization and Download**

From the Kaggle homepage, use the left sidebar to access the executed notebook:

- Click **Your Work**
- Click the notebook name

If the notebook is still running, open the **Logs** tab to monitor the textual logs printed in real time.

If the notebook has completed, open the **Output** tab to download the generated files. Depending on the enabled flags, the following archives may be produced:

- `logs.zip`
- `checkpoints.zip`

This makes them easier to download from the Kaggle output artifacts.